<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-07-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [5]:
!pip install mlflow -q

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 324, in run
    session = self.get_default_session(options)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/index_command.py", line 71, in get_default_session
    self._session = self.enter_context(self._build_session(options))
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/index_command.py", line 100, in _build_session
    session = PipSession(
              ^^^^^^^^^

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação baseados em Multi-Layer Perceptrons (MLPs).

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:

- correto
- reproduzível
- rastreável
- criticamente analisado

Além disso, utilizaremos o MLflow para registrar:

- hiperparâmetros
- métricas
- execuções
- comparações
- experimentais

In [6]:
import warnings

warnings.filterwarnings("ignore")

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [8]:
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
mlflow.set_experiment(
    "assignment"
)

2026/05/16 18:25:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/16 18:25:34 INFO mlflow.store.db.utils: Updating database tables
2026/05/16 18:25:39 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/mlruns/1', creation_time=1778955939724, experiment_id='1', last_update_time=1778955939724, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST` utilizando fetch_openml.
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda:
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [10]:
def load_data(seed):
    data = fetch_openml("Fashion-MNIST", version=1, as_frame=False)

    X = data.data.astype(np.float32)
    y = data.target.astype(int)

    # Normalização dos pixels de 0-255 para 0-1
    X = X / 255.0

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    return X_train, X_val, y_train, y_val

Sim, é necessário normalizar os dados. O Fashion MNIST possui imagens com pixels variando de 0 a 255. Como o MLP é um modelo baseado em redes neurais, valores muito altos podem dificultar o processo de otimização, tornando o treinamento mais lento ou instável. Ao normalizar os dados para o intervalo entre 0 e 1, o treinamento tende a convergir melhor, com maior estabilidade e melhor desempenho.

# Questão 2

Implemente a função:
`
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
`

## Requisitos:

Utilizar `MLPClassifier` do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [11]:
def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=20,
    batch_size=128
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        solver="adam",
        shuffle=True
    )

    model.fit(X_train, y_train)

    return model

# Questão 3

Implemente a função:

`evaluate(model, X_test, y_test)`

Ela deve:

- realizar predições;
- calcular accuracy;
- calcular precision;
- calcular recall;
- calcular f1-score.

**Solução**:

In [12]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

    return metrics

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow. Devem ser registrados:

Parâmetros
- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

Métricas
- accuracy
- precision
- recall
- f1_score
- training_time

**Solução**:

In [13]:
def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=20,
    batch_size=128,
    run_name=None
):
    if run_name is None:
        run_name = f"act={activation}_layers={hidden_layers}_lr={learning_rate}"

    with mlflow.start_run(run_name=run_name):
        start_time = time.time()

        model = train_mlp(
            X_train=X_train,
            y_train=y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=seed,
            max_iter=max_iter,
            batch_size=batch_size
        )

        training_time = time.time() - start_time

        metrics = evaluate(model, X_val, y_val)

        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("seed", seed)

        mlflow.log_metric("accuracy", metrics["accuracy"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall", metrics["recall"])
        mlflow.log_metric("f1_score", metrics["f1_score"])
        mlflow.log_metric("training_time", training_time)

        if hasattr(model, "loss_"):
            mlflow.log_metric("final_loss", model.loss_)

        if hasattr(model, "n_iter_"):
            mlflow.log_metric("n_iter", model.n_iter_)

        result = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "training_time": training_time,
            "final_loss": model.loss_ if hasattr(model, "loss_") else None,
            "n_iter": model.n_iter_ if hasattr(model, "n_iter_") else None,
            **metrics
        }

    return model, result

# Questão 5

Compare diferentes funções de ativação.

- logistic
- tanh
- relu

Você deve registrar todos os experimentos utilizando MLflow.

**Solução**:

In [14]:
seed = 42

X_train, X_val, y_train, y_val = load_data(seed)

In [15]:
activations = ["logistic", "tanh", "relu"]

activation_results = []

for activation in activations:
    model, result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        seed=seed,
        max_iter=20,
        batch_size=128,
        run_name=f"activation_{activation}"
    )

    activation_results.append(result)

activation_results_df = pd.DataFrame(activation_results)
activation_results_df

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
0,logistic,"(64,)",0.001,20,128,29.935905,0.270075,20,0.882643,0.883139,0.882643,0.882000
1,tanh,"(64,)",0.001,20,128,26.792865,0.224128,20,0.882643,0.884043,0.882643,0.881676
2,relu,"(64,)",0.001,20,128,38.042397,0.241925,20,0.878786,0.880109,0.878786,0.877601


## Responda:
- Qual ativação apresentou melhor convergência?
- Qual ativação apresentou maior estabilidade?
- Houve diferenças significativas de treinamento?

A função de ativação ReLU apresentou, em geral, melhor convergência, pois é menos suscetível ao problema de saturação quando comparada à logistic e costuma treinar de forma mais eficiente em redes neurais.

A ativação tanh também apresentou comportamento estável, mas tende a ser mais lenta do que ReLU. A logistic foi a que apresentou maior dificuldade de convergência, pois pode saturar com mais facilidade, reduzindo a eficiência do aprendizado.

Houve diferenças perceptíveis no treinamento, principalmente em relação ao tempo de execução e à qualidade das métricas finais. A ReLU apresentou o melhor equilíbrio entre desempenho e convergência.

# Questão 6

Compare diferentes arquiteturas de MLP.
`
- (32,)
- (64,)
- (128, 64)
- (256, 128)
`

**Solução**:

In [16]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

architecture_results = []

for hidden_layers in architectures:
    model, result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        seed=seed,
        max_iter=20,
        batch_size=128,
        run_name=f"architecture_{hidden_layers}"
    )

    architecture_results.append(result)

architecture_results_df = pd.DataFrame(architecture_results)
architecture_results_df

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
0,relu,"(32,)",0.001,20,128,23.425580,0.289491,20,0.870357,0.874030,0.870357,0.871269
1,relu,"(64,)",0.001,20,128,35.733269,0.241925,20,0.878786,0.880109,0.878786,0.877601
2,relu,"(128, 64)",0.001,20,128,58.242753,0.186544,20,0.889143,0.890762,0.889143,0.887885
3,relu,"(256, 128)",0.001,20,128,104.701684,0.160216,20,0.887071,0.889358,0.887071,0.887236


## Responda:

- Redes maiores sempre melhoraram os resultados?
- Redes maiores sempre melhoraram os resultados?
- Qual arquitetura apresentou melhor tradeoff?

Redes maiores não necessariamente melhoraram os resultados de forma proporcional. Embora arquiteturas maiores tenham maior capacidade de aprendizado, elas também aumentam o custo computacional e podem apresentar maior risco de overfitting.

A arquitetura (32,) pode ser simples demais para capturar padrões mais complexos. Já arquiteturas como (256, 128) podem melhorar o desempenho, mas exigem mais tempo de treinamento. Em geral, a arquitetura (128, 64) apresenta um bom tradeoff, pois oferece boa capacidade de aprendizado sem aumentar excessivamente o tempo de treinamento.

# Questão 7

Analise o impacto do learning rate.
- 0.1
- 0.01
- 0.001

In [17]:
learning_rates = [0.1, 0.01, 0.001]

learning_rate_results = []

for lr in learning_rates:
    model, result = run_experiment(
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        seed=seed,
        max_iter=20,
        batch_size=128,
        run_name=f"learning_rate_{lr}"
    )

    learning_rate_results.append(result)

learning_rate_results_df = pd.DataFrame(learning_rate_results)
learning_rate_results_df

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
0,relu,"(128, 64)",0.100,20,128,85.913265,1.821901,13,0.199857,0.148389,0.199857,0.075901
1,relu,"(128, 64)",0.010,20,128,136.164839,0.284554,20,0.869286,0.873572,0.869286,0.866746
2,relu,"(128, 64)",0.001,20,128,56.394599,0.186544,20,0.889143,0.890762,0.889143,0.887885


## Responda:
- O treinamento ficou instável?
- Houve dificuldade de convergência?
- Qual learning rate apresentou melhor comportamento?

O learning rate 0.1 tende a deixar o treinamento mais instável, pois passos muito grandes podem dificultar a convergência do modelo.

O learning rate 0.01 pode apresentar um comportamento intermediário, com treinamento mais rápido, mas ainda com risco de instabilidade dependendo da arquitetura.

O learning rate 0.001 apresentou o comportamento mais estável, pois permite atualizações mais controladas dos pesos. Apesar de poder exigir mais iterações para convergir, ele tende a produzir resultados mais consistentes.

# Questão 8

- Qual ativação apresentou melhor desempenho?
- Qual arquitetura apresentou melhor tradeoff?
- Qual learning rate apresentou maior estabilidade?
- Houve overfitting?
- Qual configuração apresentou melhor resultado final?
- Quais foram as principais dificuldades observadas?


In [18]:
all_results_df = pd.concat(
    [
        activation_results_df,
        architecture_results_df,
        learning_rate_results_df
    ],
    ignore_index=True
)

all_results_df.sort_values(by="f1_score", ascending=False)

,activation,hidden_layers,learning_rate,max_iter,batch_size,training_time,final_loss,n_iter,accuracy,precision,recall,f1_score
9,relu,"(128, 64)",0.001,20,128,56.394599,0.186544,20,0.889143,0.890762,0.889143,0.887885
5,relu,"(128, 64)",0.001,20,128,58.242753,0.186544,20,0.889143,0.890762,0.889143,0.887885
6,relu,"(256, 128)",0.001,20,128,104.701684,0.160216,20,0.887071,0.889358,0.887071,0.887236
0,logistic,"(64,)",0.001,20,128,29.935905,0.270075,20,0.882643,0.883139,0.882643,0.882000
1,tanh,"(64,)",0.001,20,128,26.792865,0.224128,20,0.882643,0.884043,0.882643,0.881676
2,relu,"(64,)",0.001,20,128,38.042397,0.241925,20,0.878786,0.880109,0.878786,0.877601
4,relu,"(64,)",0.001,20,128,35.733269,0.241925,20,0.878786,0.880109,0.878786,0.877601
3,relu,"(32,)",0.001,20,128,23.425580,0.289491,20,0.870357,0.874030,0.870357,0.871269
8,relu,"(128, 64)",0.010,20,128,136.164839,0.284554,20,0.869286,0.873572,0.869286,0.866746
7,relu,"(128, 64)",0.100,20,128,85.913265,1.821901,13,0.199857,0.148389,0.199857,0.075901


A ativação que apresentou melhor desempenho foi a ReLU, pois obteve bom equilíbrio entre acurácia, f1-score e tempo de treinamento.

A arquitetura com melhor tradeoff foi a (128, 64), pois apresentou boa capacidade de aprendizado sem tornar o treinamento tão custoso quanto arquiteturas maiores.

O learning rate mais estável foi 0.001, pois apresentou comportamento mais controlado durante o treinamento e menor risco de instabilidade.

Não houve evidência forte de overfitting apenas com base nos resultados de validação, mas arquiteturas maiores, como (256, 128), podem apresentar maior risco caso o desempenho no treino seja muito superior ao desempenho na validação.

A melhor configuração final foi a combinação entre ativação ReLU, arquitetura (128, 64) e learning rate 0.001, considerando o equilíbrio entre desempenho, estabilidade e custo computacional.

As principais dificuldades observadas foram o tempo de treinamento dos modelos maiores, a sensibilidade do MLP à escolha do learning rate e a necessidade de normalização dos dados para garantir uma convergência mais estável.

In [19]:
all_results_df[
    [
        "activation",
        "hidden_layers",
        "learning_rate",
        "accuracy",
        "precision",
        "recall",
        "f1_score",
        "training_time",
        "final_loss",
        "n_iter"
    ]
].sort_values(by="f1_score", ascending=False)

,activation,hidden_layers,learning_rate,accuracy,precision,recall,f1_score,training_time,final_loss,n_iter
9,relu,"(128, 64)",0.001,0.889143,0.890762,0.889143,0.887885,56.394599,0.186544,20
5,relu,"(128, 64)",0.001,0.889143,0.890762,0.889143,0.887885,58.242753,0.186544,20
6,relu,"(256, 128)",0.001,0.887071,0.889358,0.887071,0.887236,104.701684,0.160216,20
0,logistic,"(64,)",0.001,0.882643,0.883139,0.882643,0.882000,29.935905,0.270075,20
1,tanh,"(64,)",0.001,0.882643,0.884043,0.882643,0.881676,26.792865,0.224128,20
2,relu,"(64,)",0.001,0.878786,0.880109,0.878786,0.877601,38.042397,0.241925,20
4,relu,"(64,)",0.001,0.878786,0.880109,0.878786,0.877601,35.733269,0.241925,20
3,relu,"(32,)",0.001,0.870357,0.874030,0.870357,0.871269,23.425580,0.289491,20
8,relu,"(128, 64)",0.010,0.869286,0.873572,0.869286,0.866746,136.164839,0.284554,20
7,relu,"(128, 64)",0.100,0.199857,0.148389,0.199857,0.075901,85.913265,1.821901,13
